# Thử nghiệm HPO theo thời gian — LightGBM + chu kỳ

Nghiên cứu 30 trial Optuna với bốn rolling fold **chỉ thuộc train**. Mỗi fold giữ một cutoff embargo; metric lựa chọn là Brier trung bình không trọng số giữa fold. Validation chỉ được đọc sau khi hoàn thành search, để đối chiếu cấu hình cố định ban đầu với cấu hình được chọn. Không đọc test và không ghi đè pipeline POC.

Kết quả fold được dùng để chọn hyperparameter nên điểm tốt nhất không phải ước lượng độc lập. Validation cũng từng được sử dụng nhiều lần trong project; cần dữ liệu tương lai chưa dùng để xác nhận.

## 1. Nạp train và xây dựng feature chu kỳ

Sử dụng đúng 18 feature và 5 feature chu kỳ như POC: khoảng cách giữa các ngày có basket trong `(s-180,s]`. Không dùng giao dịch sau cutoff hoặc `QUANTITY`. Mỗi lần fit giữ nguyên số vòng lặp 200, không early stopping.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
for folder in ['.deps','.deps/hpo']:
    if (ROOT/folder).is_dir():sys.path.insert(0,str(ROOT/folder))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
import lightgbm,optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
DATA=ROOT/'artifacts'/'data_preparation';OUT=ROOT/'artifacts'/'research'/'hpo_temporal_research'
train,features=load_prepared_split(DATA,'train')
cadence=['cadence_median_gap_180d','cadence_gap_std_180d','cadence_recency_ratio','cadence_has_gap','cadence_has_two_gaps']
def active_until(cutoff):
    source=pd.read_csv(ROOT/'sample_data'/'dunnhumby'/'transaction_data.csv',usecols=['household_key','DAY'])
    return source.loc[source.DAY<=cutoff].drop_duplicates().sort_values(['household_key','DAY'])
def add_cadence(panel,active):
    parts=[]
    for s in sorted(panel.snapshot_day.unique()):
        h=panel.loc[panel.snapshot_day.eq(s),['household_key','snapshot_day','recency_days']].set_index('household_key').copy()
        d=active.loc[active.DAY.between(s-179,s)].copy();d['gap']=d.groupby('household_key').DAY.diff()
        gap=d.groupby('household_key').gap.agg(['count','median','std'])
        h[cadence[0]]=gap['median'];h[cadence[1]]=gap['std'];h[cadence[2]]=h.recency_days/h[cadence[0]]
        h[cadence[3]]=gap['count'].ge(1).reindex(h.index,fill_value=False).astype(int)
        h[cadence[4]]=gap['count'].ge(2).reindex(h.index,fill_value=False).astype(int)
        parts.append(h.reset_index()[['household_key','snapshot_day']+cadence])
    result=panel.merge(pd.concat(parts,ignore_index=True),on=['household_key','snapshot_day'],validate='one_to_one')
    assert len(result)==len(panel) and not result[cadence].isin([np.inf,-np.inf]).any().any()
    return result
train=add_cadence(train,active_until(537));model_features=features+cadence
print('Optuna:',optuna.__version__,'| LightGBM:',lightgbm.__version__,'| Train rows:',len(train))

Optuna: 5.0.0 | LightGBM: 4.7.0 | Train rows: 129834


## 2. Định nghĩa fold và search space

Các cutoff train của mỗi fold có nhãn trưởng thành trước cutoff đầu tiên của fold đánh giá. Folds lấy bốn giai đoạn 6 cutoff, có train mở rộng. Giữ nguyên cây 200 vòng; search `num_leaves` 15–63, `min_child_samples` 30–100, `reg_lambda` 0,5–10 và learning rate 0,03–0,15. Seed 42; 30 trial và cùng 4 fold cho mỗi trial. Không sử dụng validation/test trong vòng search.

In [2]:
days=sorted(train.snapshot_day.unique());folds=[]
for i,start in enumerate([20,28,36,44],1):
    tr=days[:start-1];ev=days[start:start+6]
    assert len(ev)==6 and max(tr)+7<min(ev)
    folds.append({'fold':i,'train_days':tr,'eval_days':ev})
fold_manifest=pd.DataFrame([{'fold':f['fold'],'train_end':max(f['train_days']),'eval_start':min(f['eval_days']),'eval_end':max(f['eval_days'])} for f in folds])
display(fold_manifest)
base_params={'num_leaves':15,'min_child_samples':100,'reg_lambda':0.0,'learning_rate':.05}
def make_model(params):
    return LGBMClassifier(n_estimators=200,random_state=42,verbosity=-1,n_jobs=2,**params)
def fold_briers(params):
    values=[]
    for fold in folds:
        tr=train.snapshot_day.isin(fold['train_days']);ev=train.snapshot_day.isin(fold['eval_days'])
        model=make_model(params);model.fit(train.loc[tr,model_features],train.loc[tr,'target_any_basket'])
        p=model.predict_proba(train.loc[ev,model_features])[:,1]
        values.append(brier_score_loss(train.loc[ev,'target_any_basket'],p))
    return values
baseline_folds=fold_briers(base_params)
def objective(trial):
    params={'num_leaves':trial.suggest_int('num_leaves',15,63),
            'min_child_samples':trial.suggest_int('min_child_samples',30,100),
            'reg_lambda':trial.suggest_float('reg_lambda',.5,10.,log=True),
            'learning_rate':trial.suggest_float('learning_rate',.03,.15,log=True)}
    scores=fold_briers(params)
    for i,v in enumerate(scores,1):trial.set_user_attr(f'fold_{i}_brier',v)
    return float(np.mean(scores))
study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective,n_trials=30,n_jobs=1,show_progress_bar=False)
best_params=study.best_params
print('Trial:',len(study.trials),'| Brier CV gốc:',np.mean(baseline_folds),'| Brier CV chọn:',study.best_value)
display(pd.DataFrame([base_params,best_params],index=['cấu hình gốc','được chọn']))

Trial: 30 | Brier CV gốc: 0.16358287365454532 | Brier CV chọn: 0.16340089081952872


,fold,train_end,eval_start,eval_end
0,1,306,320,355
1,2,362,376,411
2,3,418,432,467
3,4,474,488,523


,num_leaves,min_child_samples,reg_lambda,learning_rate
cấu hình gốc,15,100,0.000000,0.050000
được chọn,24,58,7.325473,0.032123


## 3. Đối chiếu validation sau khi chốt search

Hai model được fit trên toàn train (cutoff đến 537), cùng feature và số vòng. Không calibration hay chọn ngưỡng; so sánh xác suất thô bằng Brier, log loss, ROC-AUC, lift trung bình theo cutoff. Lift tính top 10% mỗi tuần và kỳ vọng nếu đồng điểm. **Khác pipeline POC bàn giao đã fit đến 439**; không đặt kết quả này ngang hàng như cùng model. Không dùng test để quyết định.

In [3]:
validation,vfeatures=load_prepared_split(DATA,'validation')
assert features==vfeatures and train.snapshot_day.max()+7<validation.snapshot_day.min()
validation=add_cadence(validation,active_until(int(validation.snapshot_day.max()))).reset_index(drop=True)
def weekly_lift(frame,p):
    results=[]
    for ix in frame.groupby('snapshot_day').indices.values():
        y=frame.target_any_basket.to_numpy()[ix];q=p[ix];k=int(np.ceil(.1*len(y)));threshold=np.partition(q,len(q)-k)[len(q)-k]
        high=q>threshold;ties=q==threshold
        precision=(y[high].sum()+(k-high.sum())*y[ties].mean())/k
        results.append(precision/y.mean() if y.mean() else np.nan)
    return np.asarray(results)
comparison=[]
for name,params in [('cấu hình gốc',base_params),('HPO',best_params)]:
    model=make_model(params);model.fit(train[model_features],train.target_any_basket)
    p=model.predict_proba(validation[model_features])[:,1]
    assert np.isfinite(p).all()
    comparison.append({'model':name,'brier':brier_score_loss(validation.target_any_basket,p),
                       'log_loss':log_loss(validation.target_any_basket,p,labels=[0,1]),
                       'roc_auc':roc_auc_score(validation.target_any_basket,p),
                       'weekly_lift_mean':np.nanmean(weekly_lift(validation,p))})
comparison=pd.DataFrame(comparison);display(comparison)

,model,brier,log_loss,roc_auc,weekly_lift_mean
0,cấu hình gốc,0.164026,0.494306,0.837327,1.791297
1,HPO,0.164038,0.494343,0.837335,1.794736


**Kết quả:** HPO đạt Brier trung bình 0,163401 trên các fold dùng để tìm kiếm, so với 0,163583 của cấu hình gốc; đây không phải so sánh độc lập. Trên validation, HPO có Brier 0,164038 và log loss 0,494343 so với 0,164026 và 0,494306 của cấu hình gốc; lift tuần lần lượt 1,794736 và 1,791297. Chênh lệch nhỏ và không đồng thuận; không ghi đè model POC dựa trên lần thử này.


## 4. Kết quả chỉ phục vụ nghiên cứu

Không tự hardcode tham số HPO vào pipeline. Kết quả tối ưu trên cùng fold đã dùng để chọn có thể lạc quan; chênh lệch nhỏ trên validation không đủ chứng minh hiệu năng tương lai. Calibration, độ ổn định và thay đổi mô hình cần đánh giá theo thiết kế riêng, với dữ liệu chưa dùng để chọn.

In [4]:
OUT.mkdir(parents=True,exist_ok=True)
study.trials_dataframe().to_csv(OUT/'trials.csv',index=False,encoding='utf-8-sig')
comparison.to_csv(OUT/'validation_comparison.csv',index=False,encoding='utf-8-sig')
fold_manifest.to_csv(OUT/'folds.csv',index=False,encoding='utf-8-sig')
(OUT/'best_params.json').write_text(json.dumps({'best_params':best_params,'best_cv_brier':study.best_value,
    'baseline_fold_brier':baseline_folds,'trials':30,'seed':42,'features':model_features,
    'lightgbm_version':lightgbm.__version__,'optuna_version':optuna.__version__},indent=2),encoding='utf-8')
print('Đã lưu:',OUT)

Đã lưu: artifacts\hpo_temporal_research
